# 02 · LangGraph Memory (MemorySaver & Threads)

Why every graph.invoke() forgets, and how checkpointers and thread_ids give an agent memory.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AfzalManihar/Agentic-AI-Learning/blob/main/02_langgraph_memory_saver.ipynb)

**Covered in this notebook**

- Topic: Memory

> Part of [Agentic AI Learning](https://github.com/AfzalManihar/Agentic-AI-Learning). Add your own API keys in Colab Secrets (key icon) before running.

#06-08-26

# Topic: Memory

In [ ]:
response = graph.invoke({'messages': [('user', 'My name is Adam')]})
response['messages'][-1].content

'Nice to meet you, Adam! How can I help you today?'

In [ ]:
# Everytime graph.invoke() starts fresh, it has no idea what you said before
response = graph.invoke({'messages': [('user', 'What is my name')]})
response['messages'][-1].content

'I’m not sure what your name is—could you let me know?'

In [ ]:
# Memory Saver
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

In [ ]:
builder = StateGraph(State) # Blank Graph
builder.add_node('assistant', assistant_node)
builder.add_node('tools', tool_node)
builder.add_edge(START, 'assistant')
builder.add_conditional_edges('assistant', should_continue)
builder.add_edge('tools', 'assistant')

graph_with_memory = builder.compile(checkpointer=memory)

In [ ]:
config1 = {'configurable': {'thread_id': 'Adam_chat'}}

In [ ]:
response = graph_with_memory.invoke({'messages': [('user', 'My name is Adam')]}, config=config1)
response['messages'][-1].content

'Nice to meet you, Adam! How can I help you today?'

In [ ]:
response = graph_with_memory.invoke({'messages': [('user', 'What is my name')]}, config=config1)
response['messages'][-1].content

'Your name is Adam.'

In [ ]:
config2 = {'configurable': {'thread_id': 'Rahul_chat'}}

In [ ]:
response = graph_with_memory.invoke({'messages': [('user', 'What is my name')]}, config=config2)
response['messages'][-1].content

'I’m not sure what your name is—could you let me know?'

In [ ]:
config3 = {"configurable": {"thread_id": "conversation_3"}}

In [ ]:
turns = [
    "I am a data scientist working in Bangalore.",
    "I love Python and I mostly work with tabular data.",
    "Based on what I told you, suggest one tool I should learn next."
]

In [ ]:
for message in turns:
    response = graph_with_memory.invoke(
        {"messages": [("user", message)]},
        config=config3
    )
    print(f"User: {message}")
    print(f"Agent: {response['messages'][-1].content}")
    print("---")

User: I am a data scientist working in Bangalore.
Agent: Nice to meet you! Bangalore has become a vibrant hub for data science, with plenty of opportunities ranging from AI research to industry‑focused analytics. How can I help you today? Are you looking for resources, career advice, project ideas, or something else?
---
User: I love Python and I mostly work with tabular data.
Agent: That’s great! Python is the go‑to language for tabular data, and Bangalore’s data‑science ecosystem is buzzing with tools, communities, and projects. Below are a few “quick‑start” pointers that might help you sharpen your workflow, discover new libraries, or find inspiration for your next project.

---

## 1. Core Python Stack for Tabular Data

| Library | What it does | Why it matters |
|---------|--------------|----------------|
| **pandas** | DataFrames, I/O, groupby, pivot, etc. | The de‑facto standard for tabular manipulation. |
| **numpy** | Fast array ops, broadcasting | Underpins pandas and many ML

In [ ]:
# Challenge 1 (Easy)
# Start a conversation where you tell the agent your favourite programming language and
# favourite food in a separate messages.
# Then in a third message, ask it to repeat both back to you.

In [ ]:
config4 = {"configurable": {"thread_id": "conversation_4"}}

In [ ]:
turns = [
    "My favourite programming language is python.",
    "My favourite food is biryani.",

]

In [ ]:
config4 = {"configurable": {"thread_id": "conversation_4"}}

turns = [
    "My favourite programming language is python.",
    "My favourite food is biryani.",
    "What are my favourite programming language and food?"
]

for message in turns:
    response = graph_with_memory.invoke(
        {"messages": [("user", message)]},
        config=config4
    )
    print(f"User: {message}")
    print(f"Agent: {response['messages'][-1].content}")
    print("---")

User: My favourite programming language is python.
Agent: That’s great! Python’s versatility and readability make it a favorite for many developers. What do you enjoy most about it—its libraries, its simplicity, or something else?
---
User: My favourite food is biryani.
Agent: Biryani is a classic! The fragrant rice, spices, and tender meat (or veggies) make it a real crowd‑pleaser. Do you have a favorite style—Hyderabadi, Lucknowi, or maybe a homemade recipe you love?
---
User: What are my favourite programming language and food?
Agent: Your favorite programming language is **Python**, and your favorite food is **biryani**.
---


In [ ]:
# Challenge 2 (Medium)
# Simulate two different students having separate conversations with the agent.
# Student A says their name is Priya and she studies biology.
# Student B says his name is Ravi and he studies physics.
# Then ask both "What do I study?" and verify the agent keeps them separate.

In [ ]:
# Student A conversation
config5 = {"configurable": {"thread_id": "priya_chat"}}

In [ ]:
response = graph_with_memory.invoke(
    {"messages": [("user", "My name is Priya and I study biology.")]},
    config=config5
)
response['messages'][-1].content

'Nice to meet you, Priya! Biology is such a fascinating field—what area are you most interested in? Are you working on a particular project or just exploring the basics right now?'

In [ ]:
response = graph_with_memory.invoke(
    {"messages": [("user", "What is my name and What do I study?")]},
    config=config5
)
response['messages'][-1].content

'You’re Priya, and you study biology.'

In [ ]:
# Student B conversation
config6 = {"configurable": {"thread_id": "ravi_chat"}}

In [ ]:
response = graph_with_memory.invoke(
    {"messages": [("user", "My name is Ravi and I study physics.")]},
    config=config6
)
response['messages'][-1].content

'Nice to meet you, Ravi! Physics is a fascinating field—what area are you most interested in? If you have any questions, need help with a concept, or just want to chat about the latest discoveries, feel free to let me know!'

In [ ]:
response = graph_with_memory.invoke(
    {"messages": [("user", "What is my name and What do I study?")]},
    config=config6
)
response['messages'][-1].content

'You’re Ravi, and you study physics.'

In [ ]:
# Challenge 3 (Hard)
# Build a simple number tracking conversation.
# Tell the agent three numbers across three separate messages.
# Then ask it to add all three numbers together. The agent must use the add tool and its memory to answer.

In [ ]:
# Solution
# Make sure your graph_with_memory has the add tool bound

config = {"configurable": {"thread_id": "number_track"}}

messages = [
    "Remember this number: 42",
    "Remember this number: 17",
    "Remember this number: 91",
    "Add all three numbers I gave you."
]

for msg in messages:
    response = graph_with_memory.invoke(
        {"messages": [("user", msg)]},
        config=config
    )
    print(f"User: {msg}")
    print(f"Agent: {response['messages'][-1].content}")
    print("---")

User: Remember this number: 42
Agent: Got it—42 is now remembered. Let me know if you need anything else!
---
User: Remember this number: 17
Agent: Got it—42 and 17 are now remembered. Let me know if you need anything else!
---
User: Remember this number: 91
Agent: Got it—42, 17, and 91 are all remembered. Let me know if you need anything else!
---
User: Add all three numbers I gave you.
Agent: The sum of 42, 17, and 91 is **150**.
---
